# match_case
`match`/`case` gives Python a structural pattern-matching tool for multi-branch logic. It can make command routing and shape-based handling cleaner, but only when the branching problem actually fits pattern matching better than plain `if`/`elif`.

## 1. Problem First
Why does this matter in real systems?
- CLIs often dispatch on command names and argument shapes.
- Event processors may branch based on payload structure.
- A good `match` can clarify routing; a forced one can confuse readers unfamiliar with it.

In [1]:
command = "deploy"
match command:
    case "deploy":
        print("start deployment")
    case "rollback":
        print("start rollback")
    case _:
        print("unknown command")

start deployment


## 2. Minimal Concept
Core syntax:
- `match value:` starts pattern matching.
- `case pattern:` defines branches.
- `case _:` is the wildcard fallback.

## 3. Mental Model
How Python actually behaves
`match` checks patterns in order, similar to `elif`, but the patterns can express shapes, literals, and destructuring. The power is not just multi-way branching; it is matching structure and binding pieces of data when a shape fits.

In [2]:
event = {"type": "error", "code": 500}
match event:
    case {"type": "error", "code": code}:
        print(f"error code {code}")
    case _:
        print("other event")

error code 500


## 4. Internal Mechanics
Pattern matching still checks cases top to bottom, but the checks can include destructuring and guards. It does not magically replace all branching; it simply gives you a richer branch language.

In [3]:
import dis

def route(command):
    match command:
        case "deploy":
            return "deploying"
        case "rollback":
            return "rolling back"
        case _:
            return "unknown"

dis.dis(route)
print(route("deploy"))

  3           0 RESUME                   0

  4           2 LOAD_FAST                0 (command)

  5           4 COPY                     1
              6 LOAD_CONST               1 ('deploy')
              8 COMPARE_OP              40 (==)
             12 POP_JUMP_IF_FALSE        2 (to 18)
             14 POP_TOP

  6          16 RETURN_CONST             2 ('deploying')

  7     >>   18 LOAD_CONST               3 ('rollback')
             20 COMPARE_OP              40 (==)
             24 POP_JUMP_IF_FALSE        1 (to 28)

  8          26 RETURN_CONST             4 ('rolling back')

  9     >>   28 NOP

 10          30 RETURN_CONST             5 ('unknown')
deploying


## 5. Edge Cases
Where it breaks:
- Readers may misread capture patterns as comparisons.
- A broad pattern placed early can shadow later cases.
- `match` can be overused for simple logic that `if` handles more clearly.
- Teams on older Python versions cannot use it.

In [4]:
value = [1, 2]
match value:
    case [first, second]:
        print(first, second)
    case _:
        print("no match")

1 2


## 6. Performance Thinking
- Performance is usually not the deciding factor between `match` and `if`.
- The bigger question is whether structural matching makes the logic easier to understand.
- For simple literal dispatch, dictionaries or plain `if` may still be clearer.

## 7. Real Use Case
A CLI router can dispatch commands and structured arguments with cleaner intent than a long `if` ladder.

In [5]:
request = ("deploy", "prod")
match request:
    case ("deploy", environment):
        print(f"deploy to {environment}")
    case ("rollback", environment):
        print(f"rollback in {environment}")
    case _:
        print("unsupported request")

deploy to prod


## 8. Anti-Pattern
What beginners do wrong:
- Use `match` just because it is new.
- Replace a simple `if` with patterns that are harder to read.
- Forget that case order still matters.

In [6]:
status = 503
match status:
    case code if code >= 500:
        print("retry")
    case code if code >= 400:
        print("fail")
    case _:
        print("ok")

retry


## 9. Interview Signals
Questions FAANG asks:
- When is `match` better than `if`/`elif`?
- What kinds of structures can Python pattern match on?
- Why can capture patterns be confusing?
- How does case ordering still matter?

## 10. Exercise (Non-trivial)
Design a command router for a CLI tool that handles tuples like `(command, environment)`, dictionaries with typed payloads, and unknown shapes. Decide where `match` improves clarity and where plain `if` should stay.

In [7]:
def route_request(request):
    match request:
        case ("deploy", env):
            return f"deploy:{env}"
        case _:
            return "unknown"

# Task:
# 1. Add rollback and validate branches.
# 2. Handle malformed request shapes.
# 3. Explain why match is or is not the right tool here.